# Notebook 1 of 3: DTW event tables and cumulative-curve calculations (163 original Northeast wells)


In [1]:
# ===== CELL 1: SETTINGS =====
import os, io, time, glob
import numpy as np, pandas as pd, geopandas as gpd, requests
from dtw import dtw                                   

# ---- input files ----
WELL_SITES   = r"C:\Users\romin\OneDrive\Groundwater\RpSy Data\Site information for all selected wells.xlsx"
STATES_PATH  = r"C:\Users\romin\OneDrive\Groundwater\cb_2022_us_state_500k"
RECHARGE_DIR = "C:/Users/romin/OneDrive/Groundwater/RpSy Data"     
DAYMET_DIR   = "./daymet"                                          
# ---- output files----
EVENT_DIR         = "dtw_event_tables_rpsy"                         
WELLS_CSV         = "wells_163.csv"                                
GAPS_ANNUAL_CSV   = "cumulative_gaps_annual_163.csv"                
GAPS_SEASONAL_CSV = "cumulative_gaps_seasonal_163.csv"              
SHARES_CSV        = "seasonal_shares_163.csv"                       

# ---- which wells ----
ORIGINAL_FLAG_CSV = "rpsy_all_summary.csv"          
ORIGINAL_IDS_FALLBACK = "master_well_summary.csv"   
NE_STATES = ["Connecticut", "Maine", "Massachusetts", "New Hampshire", "Rhode Island",
             "Vermont", "New Jersey", "New York", "Pennsylvania"]
MIN_RECORD_YEARS = 5                                                # RpSy record must be at least this long
DAYMET_START_YEAR, DAYMET_END_YEAR = 1980, 2024

# ---- DTW pairing (storms = consecutive days above 2.5 mm/day; recharge starts 0-5 days after the storm) ----
PRECIP_THRESHOLD = 2.5                                              # mm/day
MIN_LAG, MAX_LAG = 0, 5                                             # days recharge may lag the precipitation

# ---- cumulative curves (storms under 5 mm excluded) ----
MIN_STORM  = 5                                                      # mm per storm
MIN_EVENTS = 20                                                     # minimum storms for a well (or well-season) to get a curve
GRID_STEP  = 0.01                                                   # curves are read at 1%, 2%, ... 99% of cumulative precipitation
SEASONS = ["Winter", "Spring", "Summer", "Fall"]
MONTH2SEASON = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
                6: "Summer", 7: "Summer", 8: "Summer", 9: "Fall", 10: "Fall", 11: "Fall"}
GRID = np.round(np.arange(GRID_STEP, 1 - GRID_STEP / 2, GRID_STEP), 4)
GCOLS = [f"gap_{int(round(g * 100)):02d}" for g in GRID]            # gap_01 ... gap_99 (gap_25, gap_50, gap_75 are the poster marks)

os.makedirs(EVENT_DIR, exist_ok=True); os.makedirs(DAYMET_DIR, exist_ok=True)

Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



In [8]:
# ===== CELL 2: THE 163 WELLS =====
def load_sites(path=WELL_SITES):
    df = pd.read_excel(path)
    return df.rename(columns={"ID": "usgs_id", "Lat": "lat", "Long": "lon", "depth (m)": "depth"})

def select_ne_wells(df_sites, states_shp=STATES_PATH):
    states = gpd.read_file(states_shp)
    ne = states[states["NAME"].isin(NE_STATES)].set_crs(epsg=4326, allow_override=True)
    gdf = gpd.GeoDataFrame(df_sites, geometry=gpd.points_from_xy(df_sites["lon"], df_sites["lat"]), crs="EPSG:4326")
    joined = gpd.sjoin(gdf, ne[["NAME", "geometry"]], how="inner", predicate="within")
    return joined.rename(columns={"NAME": "state"}).drop(columns=["index_right"])

# Northeast wells from the site list
df_ne_wells = select_ne_wells(load_sites())
df_ne_wells["usgs_id"] = df_ne_wells["usgs_id"].astype(str)

# length of each well's RpSy record
start, end = [], []
for wid in df_ne_wells["usgs_id"]:
    f = f"{RECHARGE_DIR}/{wid}.csv"
    if not os.path.exists(f):
        start.append(pd.NaT); end.append(pd.NaT); continue
    d = pd.to_datetime(pd.read_csv(f, usecols=["Date"])["Date"])
    start.append(d.min()); end.append(d.max())
df_ne_wells["start_date"], df_ne_wells["end_date"] = start, end
df_ne_wells["record_length_yr"] = (df_ne_wells["end_date"] - df_ne_wells["start_date"]).dt.days / 365.25

wells = df_ne_wells[df_ne_wells["record_length_yr"] >= MIN_RECORD_YEARS].copy()
wells = wells.rename(columns={"usgs_id": "well_id", "depth": "well_depth_m"})
wells = wells[["well_id", "lat", "lon", "state", "well_depth_m", "start_date", "end_date", "record_length_yr"]]
wells = wells.drop_duplicates("well_id").reset_index(drop=True)
n_rule = len(wells)

# keep only the original 163 (the 5-year rule also picks up wells added later)
orig = None
if os.path.exists(ORIGINAL_FLAG_CSV):
    t = pd.read_csv(ORIGINAL_FLAG_CSV, dtype={"well_id": str})
    orig = set(t.loc[t["original_163"].astype(str).str.lower().isin(["true", "1", "1.0"]), "well_id"])
elif os.path.exists(ORIGINAL_IDS_FALLBACK):
    orig = set(pd.read_csv(ORIGINAL_IDS_FALLBACK, dtype={"well_id": str})["well_id"])

if orig is None:
    print("No file with the original wells found, so only the 5-year rule was used.")
else:
    dropped = sorted(set(wells["well_id"]) - orig)
    not_found = sorted(orig - set(wells["well_id"]))
    wells = wells[wells["well_id"].isin(orig)].reset_index(drop=True)
    print(f"5-year rule found {n_rule} wells; kept the {len(wells)} original ones, dropped {len(dropped)}")
    if dropped: print("dropped:", dropped)
    if not_found: print("original wells the rule did not find:", not_found)

wells.to_csv(WELLS_CSV, index=False)                                    # new file
print(f"\n{len(wells)} wells saved to {WELLS_CSV}")
print(wells["state"].value_counts().to_string())

5-year rule found 163 wells; kept the 163 original ones, dropped 0

163 wells saved to wells_163.csv
state
New Jersey       45
Pennsylvania     39
New York         34
Massachusetts    21
Maine            11
Connecticut       5
Rhode Island      4
Vermont           3
New Hampshire     1


In [9]:
# ===== CELL 3: DAILY PRECIPITATION =====
def fetch_daymet(lat, lon, start_year=DAYMET_START_YEAR, end_year=DAYMET_END_YEAR, retries=3):
    url = "https://daymet.ornl.gov/single-pixel/api/data"
    params = dict(lat=round(lat, 5), lon=round(lon, 5), vars="prcp",
                  start=f"{start_year}-01-01", end=f"{end_year}-12-31", format="csv")
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=120)
            r.raise_for_status()
            lines = r.text.splitlines()
            i0 = next(i for i, l in enumerate(lines) if l.startswith("year,yday"))
            return pd.read_csv(io.StringIO("\n".join(lines[i0:])))
        except (requests.RequestException, StopIteration) as e:
            if attempt == retries - 1:
                print(f"  Daymet failed ({lat}, {lon}): {e}")
                return None
            time.sleep(5 * (attempt + 1))

def ensure_daymet(wid, lat, lon):
    p = f"{DAYMET_DIR}/{wid}.csv"
    if os.path.exists(p):
        return True
    dm = fetch_daymet(lat, lon)
    if dm is None:
        return False
    dm.to_csv(p, index=False); time.sleep(1)
    return True

missing = [w for w in wells["well_id"] if not os.path.exists(f"{DAYMET_DIR}/{w}.csv")]
print(f"Daymet files already here: {len(wells) - len(missing)} | to download: {len(missing)}")
for _, w in wells[wells["well_id"].isin(missing)].iterrows():
    if not ensure_daymet(w["well_id"], w["lat"], w["lon"]):
        print(w["well_id"], "daymet failed")

Daymet files already here: 163 | to download: 0


In [10]:
# ===== CELL 4: DTW PAIRING FUNCTIONS =====
# Recharge (query) is warped onto precipitation (reference) with a causal window: recharge may sit MIN_LAG..MAX_LAG days AFTER the rain.
# Both series are divided by their own standard deviation first so neither dominates the alignment.
def get_events(values, threshold):
    """Start/end index pairs for runs of consecutive values above the threshold (= one storm)."""
    above = (values > threshold).astype(int)
    padded = np.concatenate(([0], above, [0]))
    diff = np.diff(padded)
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0] - 1
    return starts, ends

def precip_recharge_event_table(df, precip_col, recharge_col, alignment, precip_threshold=0.0):
    """One row per storm: its precipitation, plus the recharge that the DTW alignment matched to it (NaN = unpaired storm)."""
    dates = df.index
    precip_vals = df[precip_col].values
    recharge_vals = df[recharge_col].values
    starts, ends = get_events(precip_vals, precip_threshold)

    rows = []
    prev_recharge_idx = None
    for s, e in zip(starts, ends):
        precip_idx = np.arange(s, e + 1)
        mask = np.isin(alignment.index2, precip_idx)
        recharge_idx = np.unique(alignment.index1[mask])

        if prev_recharge_idx is not None and recharge_idx.size > 0:
            recharge_idx = recharge_idx[recharge_idx > prev_recharge_idx.max()]     # a recharge day is only given to one storm

        row = {"precip_start_date": dates[s], "precip_end_date": dates[e],
               "precip_total": precip_vals[precip_idx].sum(), "precip_peak": precip_vals[precip_idx].max()}
        if recharge_idx.size == 0:
            row.update({"recharge_start_date": pd.NaT, "recharge_end_date": pd.NaT,
                        "recharge_total": np.nan, "recharge_peak": np.nan, "n_recharge_days": 0})
        else:
            row.update({"recharge_start_date": dates[recharge_idx.min()], "recharge_end_date": dates[recharge_idx.max()],
                        "recharge_total": recharge_vals[recharge_idx].sum(), "recharge_peak": recharge_vals[recharge_idx].max(),
                        "n_recharge_days": recharge_idx.size})
        rows.append(row)
        prev_recharge_idx = recharge_idx if recharge_idx.size > 0 else prev_recharge_idx
    return pd.DataFrame(rows)

def run_dtw_pipeline(site_id, recharge_dir=RECHARGE_DIR, daymet_dir=DAYMET_DIR, out_dir=EVENT_DIR,
                     precip_threshold=PRECIP_THRESHOLD, min_lag=MIN_LAG, max_lag=MAX_LAG):
    recharge_path, precip_path = f"{recharge_dir}/{site_id}.csv", f"{daymet_dir}/{site_id}.csv"
    if not os.path.exists(recharge_path) or not os.path.exists(precip_path):
        return None, "missing file"

    df_recharge, df_precip = pd.read_csv(recharge_path), pd.read_csv(precip_path)
    df_precip["Date"] = pd.to_datetime(df_precip["year"] * 1000 + df_precip["yday"], format="%Y%j")
    df_recharge["Date"] = pd.to_datetime(df_recharge["Date"])
    df_recharge.set_index("Date", inplace=True)
    df_precip.set_index("Date", inplace=True)

    df = pd.merge(df_recharge, df_precip, on="Date", how="inner")
    df.drop(columns=["year", "yday"], inplace=True, errors="ignore")
    if len(df) < 30:
        return None, "too short"

    recharge, precip = df["RpSy (m)"].values, df["prcp (mm/day)"].values
    if np.std(recharge) == 0 or np.std(precip) == 0:
        return None, "zero variance"
    recharge_norm, precip_norm = recharge / np.std(recharge), precip / np.std(precip)

    def causal_window(iw, jw, query_size, reference_size, min_lag=min_lag, max_lag=max_lag, **kwargs):
        lag = iw - jw
        ok = lag >= min_lag
        if max_lag is not None:
            ok = ok & (lag <= max_lag)
        return ok

    try:
        alignment = dtw(recharge_norm, precip_norm, step_pattern="symmetric2", window_type=causal_window,
                        window_args={"min_lag": min_lag, "max_lag": max_lag}, keep_internals=True,
                        open_begin=False, open_end=False)
    except Exception as e:
        return None, f"dtw failed: {e}"

    event_table = precip_recharge_event_table(df, "prcp (mm/day)", "RpSy (m)", alignment, precip_threshold=precip_threshold)
    event_table.to_csv(f"{out_dir}/{site_id}_dtw_event_table.csv", index=False)
    return event_table, "ok"

In [11]:
# ===== CELL 5: RUN DTW FOR EVERY WELL (resumable: wells that already have an event table are skipped) =====
done, failed = 0, {}
for i, wid in enumerate(wells["well_id"]):
    print(f"\rWell {i + 1}/{len(wells)}", end="", flush=True)
    if os.path.exists(f"{EVENT_DIR}/{wid}_dtw_event_table.csv"):
        done += 1; continue
    table, status = run_dtw_pipeline(wid)
    if table is not None: done += 1
    else: failed[wid] = status
print(f"\n\nEvent tables ready: {done} of {len(wells)} | failed: {len(failed)}")
if failed:
    from collections import Counter
    print("Reasons:", dict(Counter(failed.values())))
    print(failed)

Well 163/163

Event tables ready: 163 of 163 | failed: 0


In [12]:
# ===== CELL 6: CUMULATIVE PRECIPITATION vs CUMULATIVE RECHARGE, DISTANCE FROM THE 1:1 LINE =====
# Storms >= MIN_STORM are ranked smallest -> largest. x = cumulative fraction of precipitation, y = cumulative fraction of recharge.
# A storm with no DTW-matched recharge counts as 0 recharge. Distance from the 1:1 line = y - x (negative = large storms dominate).
def read_events(wid):
    cols = ["precip_start_date", "recharge_start_date"]
    ev = pd.read_csv(f"{EVENT_DIR}/{wid}_dtw_event_table.csv", parse_dates=cols)
    ev = ev[ev["precip_total"] >= MIN_STORM].copy()
    ev["paired"] = ev["recharge_total"].notna()
    ev["recharge_total"] = ev["recharge_total"].fillna(0)
    ev["season"] = ev["precip_start_date"].dt.month.map(MONTH2SEASON)
    return ev

def curve(ev):
    """(x, y) of the cumulative curve, starting at (0, 0)."""
    e = ev.sort_values("precip_total")
    x = np.concatenate(([0], (e["precip_total"].cumsum() / e["precip_total"].sum()).values))
    y = np.concatenate(([0], (e["recharge_total"].cumsum() / e["recharge_total"].sum()).values))
    return x, y

def gaps(ev):
    """Distance from the 1:1 line at every GRID point, or None if the well/season is too thin."""
    if len(ev) < MIN_EVENTS or ev["recharge_total"].sum() == 0:
        return None
    x, y = curve(ev)
    return np.interp(GRID, x, y) - GRID

def pairing_stats(ev):
    """How well DTW paired this well's storms (used later to check the depth-to-water result)."""
    e = ev.sort_values("precip_total")
    third = len(e) // 3
    paired = e[e["paired"]]
    lag = (paired["recharge_start_date"] - paired["precip_start_date"]).dt.days
    return dict(n_events=len(e), frac_unpaired=1 - e["paired"].mean(),
                unpaired_small=1 - e.iloc[:third]["paired"].mean(), unpaired_large=1 - e.iloc[-third:]["paired"].mean(),
                med_storm_paired=paired["precip_total"].median(), med_storm_unpaired=e.loc[~e["paired"], "precip_total"].median(),
                med_lag_days=lag.median(), med_recharge_days=paired["n_recharge_days"].median())

annual_rows, seasonal_rows, share_rows = [], [], []
for wid in wells["well_id"]:
    f = f"{EVENT_DIR}/{wid}_dtw_event_table.csv"
    if not os.path.exists(f): continue
    ev = read_events(wid)
    if len(ev) < MIN_EVENTS: continue

    g = gaps(ev)                                                              # annual curve (all seasons together)
    annual_rows.append({"well_id": wid, **pairing_stats(ev), **dict(zip(GCOLS, g if g is not None else [np.nan] * len(GRID)))})

    for s in SEASONS:                                                          # one curve per season
        es = ev[ev["season"] == s]
        g = gaps(es)
        if g is not None:
            seasonal_rows.append({"well_id": wid, "season": s, "n_events": len(es), **dict(zip(GCOLS, g))})

    tot_p, tot_r = ev["precip_total"].sum(), ev["recharge_total"].sum()      # each season's share of the well's total
    if tot_r > 0:
        for s in SEASONS:
            es = ev[ev["season"] == s]
            share_rows.append({"well_id": wid, "season": s, "recharge_share": es["recharge_total"].sum() / tot_r,
                               "precip_share": es["precip_total"].sum() / tot_p})

annual, seasonal, shares = pd.DataFrame(annual_rows), pd.DataFrame(seasonal_rows), pd.DataFrame(share_rows)
annual.to_csv(GAPS_ANNUAL_CSV, index=False); seasonal.to_csv(GAPS_SEASONAL_CSV, index=False); shares.to_csv(SHARES_CSV, index=False)   # new files
print(f"annual curves: {annual['gap_50'].notna().sum()} wells | seasonal curves: {len(seasonal)} well-seasons "
      f"({seasonal['well_id'].nunique()} wells) | shares: {shares['well_id'].nunique()} wells")
print(f"saved: {GAPS_ANNUAL_CSV}, {GAPS_SEASONAL_CSV}, {SHARES_CSV}")

annual curves: 163 wells | seasonal curves: 652 well-seasons (163 wells) | shares: 163 wells
saved: cumulative_gaps_annual_163.csv, cumulative_gaps_seasonal_163.csv, seasonal_shares_163.csv


In [13]:
# ===== CELL 7: QUICK CHECKS AGAINST THE POSTER =====
print("Median distance from the 1:1 line, by season (negative = large storms dominate; poster: summer ~ -0.10 at 50%)")
print(seasonal.groupby("season")[["gap_25", "gap_50", "gap_75"]].median().reindex(SEASONS).round(3).to_string())

print("\nMedian share of the well's annual total (poster: summer ~18% of recharge from ~29% of precip, winter ~29% from ~21%)")
print((shares.groupby("season")[["recharge_share", "precip_share"]].median().reindex(SEASONS) * 100).round(1).to_string())

print("\nWhere is each season's median curve furthest from the 1:1 line?")
for s in SEASONS:
    med = seasonal[seasonal["season"] == s][GCOLS].median()
    k = med.abs().idxmax()
    print(f"  {s:7s} at {int(k.split('_')[1])}% of precipitation: {med[k]:.3f}")

story = "420828076484601"
if story in set(annual["well_id"]):
    r = annual.set_index("well_id").loc[story]
    print(f"\nWell {story}: {int(r['n_events'])} storms | distance at 25/50/75% = {r['gap_25']:.3f} / {r['gap_50']:.3f} / {r['gap_75']:.3f}")

Median distance from the 1:1 line, by season (negative = large storms dominate; poster: summer ~ -0.10 at 50%)
        gap_25  gap_50  gap_75
season                        
Winter   0.045   0.038   0.020
Spring   0.023   0.019   0.008
Summer  -0.060  -0.107  -0.081
Fall    -0.015  -0.034  -0.016

Median share of the well's annual total (poster: summer ~18% of recharge from ~29% of precip, winter ~29% from ~21%)
        recharge_share  precip_share
season                              
Winter            28.6          21.0
Spring            29.3          24.2
Summer            17.5          29.0
Fall              23.2          26.6

Where is each season's median curve furthest from the 1:1 line?
  Winter  at 22% of precipitation: 0.051
  Spring  at 19% of precipitation: 0.027
  Summer  at 51% of precipitation: -0.107
  Fall    at 53% of precipitation: -0.039

Well 420828076484601: 1414 storms | distance at 25/50/75% = -0.046 / -0.084 / -0.062
